# Structured Prediction: CRFs and Sequence Labeling

## Learning Objectives

1. **Contrast** independent classification with structured prediction
2. **Derive** the Conditional Random Field (CRF) model
3. **Implement** the Viterbi algorithm for CRF decoding
4. **Apply** to sequence labeling (POS tagging, NER)

## Why Structured Prediction?

In standard classification: $\hat{y} = \arg\max_y P(y | x)$ — each output independent.

In **structured prediction**, the output $y = (y_1, \ldots, y_T)$ is a sequence (or tree, graph) and outputs are **dependent**:
- POS tagging: "NN JJ NN" is unlikely (adjective between two nouns)
- NER: "B-PER I-ORG" is invalid (person tag then org continuation)

**Key idea:** model the joint distribution $P(y_1, \ldots, y_T | x_1, \ldots, x_T)$.

## Conditional Random Fields (CRF)

A **linear-chain CRF** models:
$$P(y | x) = \frac{1}{Z(x)} \exp\left(\sum_{t=1}^T \sum_k \lambda_k f_k(y_t, y_{t-1}, x, t)\right)$$

where $f_k$ are **feature functions** and $\lambda_k$ are learned weights.

**Partition function:** $Z(x) = \sum_{y'} \exp(\text{score})$ (sum over all possible sequences)

**Comparison with HMM:**
| | HMM | Linear-chain CRF |
|--|-----|-----------------|
| Model | Generative $P(x, y)$ | Discriminative $P(y|x)$ |
| Features | Emission/transition | Arbitrary features of $x$ |
| Direction | Left-to-right | Bidirectional OK |

In [ ]:
import math
from collections import defaultdict

def logsumexp(vals):
    m = max(vals)
    return m + math.log(sum(math.exp(v-m) for v in vals))

class LinearChainCRF:
    """Simplified linear-chain CRF with unary + pairwise features."""
    def __init__(self, n_labels, feature_dim):
        import random; rng = random.Random(0)
        self.L = n_labels; self.d = feature_dim
        # Unary: weight for each (label, feature)
        self.W_unary = [[rng.gauss(0,0.1) for _ in range(feature_dim)]
                        for _ in range(n_labels)]
        # Pairwise: weight for each (prev_label, curr_label)
        self.T = [[rng.gauss(0,0.1) for _ in range(n_labels)]
                  for _ in range(n_labels)]

    def unary_score(self, y, x_t):
        return sum(self.W_unary[y][f]*x_t[f] for f in range(self.d))

    def pairwise_score(self, y_prev, y_curr):
        return self.T[y_prev][y_curr]

    def viterbi(self, X):
        """Find the best label sequence using Viterbi algorithm."""
        T = len(X); L = self.L
        # dp[t][y] = best score ending at label y at position t
        dp   = [[float('-inf')]*L for _ in range(T)]
        back = [[-1]*L for _ in range(T)]
        # Initialize t=0
        for y in range(L):
            dp[0][y] = self.unary_score(y, X[0])
        # Forward
        for t in range(1, T):
            for y in range(L):
                scores = [dp[t-1][yp] + self.pairwise_score(yp, y) for yp in range(L)]
                best_prev = max(range(L), key=lambda yp: scores[yp])
                dp[t][y] = scores[best_prev] + self.unary_score(y, X[t])
                back[t][y] = best_prev
        # Backtrack
        best_last = max(range(L), key=lambda y: dp[T-1][y])
        path = [best_last]
        for t in range(T-1, 0, -1):
            path.append(back[t][path[-1]])
        return list(reversed(path))

# POS tagging: labels = [NOUN, VERB, ADJ]
# Features: [is_capital, ends_in_ing, ends_in_ed, is_short]
label_names = ['NOUN', 'VERB', 'ADJ']
crf = LinearChainCRF(n_labels=3, feature_dim=4)

# Hand-set some weights to make it sensible
# VERB: high weight for ends_in_ing (feature 1) and ends_in_ed (feature 2)
crf.W_unary[1][1] = 2.0   # VERB if ends_in_ing
crf.W_unary[1][2] = 1.5   # VERB if ends_in_ed
crf.W_unary[0][0] = 1.5   # NOUN if starts with capital
crf.W_unary[2][3] = 1.5   # ADJ if short

# Sentence: "The running dog jumped quickly"
# Features: [capital, ends_ing, ends_ed, short]
sentence_features = [
    [0, 0, 0, 1],  # "The" (short)
    [0, 1, 0, 0],  # "running" (ends in ing)
    [0, 0, 0, 1],  # "dog" (short)
    [0, 0, 1, 0],  # "jumped" (ends in ed)
    [0, 0, 0, 0],  # "quickly"
]
words = ["The", "running", "dog", "jumped", "quickly"]

sequence = crf.viterbi(sentence_features)
print("Viterbi sequence labeling:")
for word, label_idx in zip(words, sequence):
    print(f"  {word:>10}: {label_names[label_idx]}")

## CRF Training

**Forward-Backward algorithm** computes $Z(x)$ and marginals in $O(T L^2)$.

**Gradient of log-likelihood:**
$$\frac{\partial \log P(y|x)}{\partial \lambda_k} = \sum_t f_k(y_t, y_{t-1}, x, t) - \mathbb{E}_{y'|x}\left[\sum_t f_k(y_t', y_{t-1}', x, t)\right]$$

= (empirical feature count) - (expected feature count)

**Optimization:** L-BFGS or SGD with gradient computed via forward-backward.

**CRF vs LSTM-CRF:** modern NLP uses bi-LSTM to compute rich contextual representations, then CRF for the final label sequence (LSTM-CRF). Still uses Viterbi for decoding.